# D6 — SDMX Cross-Country Analysis

## Objective

In the previous notebook, we transformed BIS SDMX data into an analysis-ready dataset and explored the time-series behaviour of a single economy. While country-level analysis is valuable, most macroeconomic research involves comparing multiple economies to identify differences, similarities, and long-term trends.

The objective of this notebook is to build a cross-country analytical workflow using official BIS SDMX data.

Using the BIS Total Credit dataset, we will:

- Retrieve data for multiple economies using a single SDMX query.
- Build a tidy panel dataset suitable for cross-country analysis.
- Compare credit indicators across economies and over time.
- Rank countries based on macroeconomic indicators.
- Analyse cross-country growth dynamics and historical trends.
- Visualise similarities and differences using comparative charts and summary tables.
- Generate concise country-level analytical profiles.

By the end of this notebook, we will have extended our SDMX workflow from analysing a single country to performing comparative macroeconomic analysis across multiple economies, following the type of exploratory work commonly carried out by economists and quantitative analysts at the Bank for International Settlements (BIS).

---

## Learning Outcomes

After completing this notebook, you will be able to:

- Retrieve and combine multiple SDMX time series into a unified panel dataset.
- Perform cross-country comparisons using official BIS macroeconomic indicators.
- Construct reusable analytical workflows for panel data.
- Rank and benchmark countries using economic indicators.
- Produce publication-ready comparative visualisations.
- Prepare multi-country datasets for subsequent statistical modelling and forecasting.

This notebook marks the transition from **single-country time-series analysis** to **cross-country macroeconomic analysis**, an essential skill in applied economic research.

- Retrieve SDMX
-        │
-        ▼
- Multiple Countries
-        │
-        ▼
- Panel Dataset
-        │
-        ▼
- Cross-country Analysis
-        │
-        ▼
- Country Ranking
-        │
-        ▼
- Comparative Visualizations
-        │
-        ▼
- Country Profiles

In [2]:
# ----------------------------------------------------
# Import Libraries
# ----------------------------------------------------

import sdmx
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# ----------------------------------------------------
# Connect to BIS
# ----------------------------------------------------

client = sdmx.Client("BIS")

print(client.source.id)
print(client.source.url)

BIS
https://stats.bis.org/api/v1


In [7]:
# ----------------------------------------------------
# Select Dataflow
# ----------------------------------------------------

DATAFLOW_ID = "WS_TC"

print(DATAFLOW_ID)

WS_TC


In [8]:
# ----------------------------------------------------
# Retrieve Dataflow
# ----------------------------------------------------

flow_response = client.dataflow()

flow = flow_response.dataflow[DATAFLOW_ID]

print(flow.id)
print(flow.name)

WS_TC
Total credit


In [11]:
# ----------------------------------------------------
# Retrieve Data Structure Definition (DSD)
# ----------------------------------------------------

dsd_response = client.datastructure(flow.structure.id)

print(type(dsd_response))
print(dsd_response.structure.keys())

<class 'sdmx.message.StructureMessage'>
dict_keys(['BIS_TOTAL_CREDIT'])


In [12]:
# ----------------------------------------------------
# Get the Data Structure Definition
# ----------------------------------------------------

dsd = dsd_response.structure["BIS_TOTAL_CREDIT"]

print(dsd.id)
print(dsd.name)

BIS_TOTAL_CREDIT
BIS long series on total credit


In [14]:
# ----------------------------------------------------
# Inspect a Dimension Object
# ----------------------------------------------------

first_dim = dsd.dimensions.components[0]

print(type(first_dim))

print("\nAvailable Attributes:")
print(dir(first_dim))

<class 'sdmx.model.common.Dimension'>

Available Attributes:
['__annotations__', '__class__', '__contains__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__post_init__', '__reduce__', '__reduce_ex__', '__replace__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_preserve', '_urn', 'annotations', 'compare', 'concept_identity', 'concept_role', 'eval_annotation', 'get_annotation', 'id', 'local_representation', 'order', 'pop_annotation', 'uri', 'urn']


In [15]:
print(first_dim)

FREQ


In [16]:
vars(first_dim)

{'concept_identity': <Concept FREQ: Frequency>,
 'local_representation': <Representation: CL_FREQ, [Facet(type=FacetType(is_sequence='false', min_length=1, max_length=1, min_value=None, max_value=None, start_value=None, end_value=None, interval=None, time_interval=None, decimals=None, pattern=None, start_time=None, end_time=None, sentinel_values=None), value=None, value_type=<FacetValueType.string: 1>)]>,
 'order': 1,
 'concept_role': None,
 '_urn': URN(package='datastructure', klass='Dimension', agency='BIS', id='BIS_TOTAL_CREDIT', version='2.0', item_id='FREQ')}

In [17]:
# ----------------------------------------------------
# Inspect the Concept
# ----------------------------------------------------

first_dim = dsd.dimensions.components[0]

print(first_dim.concept_identity)

print(type(first_dim.concept_identity))

print(dir(first_dim.concept_identity))

FREQ
<class 'sdmx.model.common.Concept'>
['__annotations__', '__class__', '__class_getitem__', '__contains__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__post_init__', '__reduce__', '__reduce_ex__', '__replace__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_description', '_name', '_preserve', '_repr_kw', '_urn', 'annotations', 'append_child', 'child', 'compare', 'core_representation', 'description', 'eval_annotation', 'get_annotation', 'get_child', 'get_scheme', 'hierarchical_id', 'id', 'iso_concept', 'name', 'parent', 'pop_annotation', 'uri', 'urn']


In [18]:
print(first_dim.concept_identity.name)

Frequency


In [19]:
# ----------------------------------------------------
# Explore Dimensions
# ----------------------------------------------------

dimensions = []

for i, dim in enumerate(dsd.dimensions.components, start=1):

    codelist = None

    if (
        dim.local_representation is not None
        and dim.local_representation.enumerated is not None
    ):
        codelist = dim.local_representation.enumerated.id

    dimensions.append({

        "Position": i,

        "Dimension": dim.id,

        "Dimension Name": dim.concept_identity.name,

        "Codelist": codelist

    })

dimensions_df = pd.DataFrame(dimensions)

dimensions_df

,Position,Dimension,Dimension Name,Codelist
0,1,FREQ,Frequency,CL_FREQ
1,2,BORROWERS_CTY,Borrowers' country,CL_AREA
2,3,TC_BORROWERS,Borrowing sector,CL_TC_BORROWERS
3,4,TC_LENDERS,Lending sector,CL_TC_LENDERS
4,5,VALUATION,Valuation method,CL_VALUATION
5,6,UNIT_TYPE,Unit type,CL_BIS_UNIT
6,7,TC_ADJUST,Adjustment,CL_ADJUST
7,8,TIME_PERIOD,Time period or range,None


In [20]:
# ----------------------------------------------------
# Build Cross-Country Query
# ----------------------------------------------------

countries = [
    "IN",
    "CN",
    "US",
    "JP",
    "DE",
    "BR"
]

query_key = (
    "Q."
    + "+".join(countries)
    + ".P.A.M.770.A"
)

print(query_key)

Q.IN+CN+US+JP+DE+BR.P.A.M.770.A


In [22]:
# ----------------------------------------------------
# Retrieve Cross-Country Data
# ----------------------------------------------------

response = client.data(
    DATAFLOW_ID,
    key=query_key
)

dataset = response.data[0]

print(type(dataset))
print("Number of Series      :", len(dataset.series))
print("Number of Observations:", len(dataset.obs))

xml.Reader got no structure=… argument for StructureSpecificData


<class 'sdmx.model.v21.StructureSpecificDataSet'>
Number of Series      : 6
Number of Observations: 1399


In [23]:
# ----------------------------------------------------
# Inspect Retrieved Series
# ----------------------------------------------------

print("Series Keys:\n")

for i, key in enumerate(dataset.series.keys(), start=1):

    print(f"{i}. {key}")

    if i == 6:
        break

Series Keys:

1. (BORROWERS_CTY=US, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=770, TC_ADJUST=A, FREQ=Q, TITLE_TS=United States - Credit to Private non-financial sector from All sectors at Market value - Percentage of GDP - Adjusted for breaks, UNIT_MULT=0, UNIT_MEASURE=367)
2. (BORROWERS_CTY=BR, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=770, TC_ADJUST=A, FREQ=Q, TITLE_TS=Brazil - Credit to Private non-financial sector from All sectors at Market value - Percentage of GDP - Adjusted for breaks, UNIT_MULT=0, UNIT_MEASURE=367)
3. (BORROWERS_CTY=JP, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=770, TC_ADJUST=A, FREQ=Q, TITLE_TS=Japan - Credit to Private non-financial sector from All sectors at Market value - Percentage of GDP - Adjusted for breaks, UNIT_MULT=0, UNIT_MEASURE=367)
4. (BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=770, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market v

In [24]:
# ----------------------------------------------------
# Verify Countries Returned
# ----------------------------------------------------

returned_countries = []

for series_key in dataset.series.keys():

    returned_countries.append(
        series_key.values["BORROWERS_CTY"].value
    )

print(sorted(returned_countries))

['BR', 'CN', 'DE', 'IN', 'JP', 'US']


In [25]:
# ----------------------------------------------------
# Analysis Configuration
# ----------------------------------------------------

COUNTRIES = [
    "IN",  # India
    "CN",  # China
    "US",  # United States
    "JP",  # Japan
    "DE",  # Germany
    "BR"   # Brazil
]

FREQUENCY = "Q"
BORROWER = "P"
LENDER = "A"
VALUATION = "M"
UNIT = "770"          # Percentage of GDP
ADJUSTMENT = "A"      # Adjusted for breaks

In [28]:
# ----------------------------------------------------
# Build SDMX Query
# ----------------------------------------------------

query_key = (
    f"{FREQUENCY}."
    f"{'+'.join(COUNTRIES)}."
    f"{BORROWER}."
    f"{LENDER}."
    f"{VALUATION}."
    f"{UNIT}."
    f"{ADJUSTMENT}"
)

print("Query Key:")
print(query_key)

Query Key:
Q.IN+CN+US+JP+DE+BR.P.A.M.770.A


In [29]:
# ----------------------------------------------------
# Analysis Summary
# ----------------------------------------------------

config = pd.DataFrame({

    "Parameter": [
        "Countries",
        "Frequency",
        "Borrower",
        "Lender",
        "Valuation",
        "Unit",
        "Adjustment"
    ],

    "Selection": [
        ", ".join(COUNTRIES),
        FREQUENCY,
        BORROWER,
        LENDER,
        VALUATION,
        UNIT,
        ADJUSTMENT
    ]

})

config

,Parameter,Selection
0,Countries,"IN, CN, US, JP, DE, BR"
1,Frequency,Q
2,Borrower,P
3,Lender,A
4,Valuation,M
5,Unit,770
6,Adjustment,A


In [30]:
# ----------------------------------------------------
# Retrieve Cross-Country Data
# ----------------------------------------------------

response = client.data(
    DATAFLOW_ID,
    key=query_key
)

dataset = response.data[0]

print(type(dataset))
print("Number of Series      :", len(dataset.series))
print("Number of Observations:", len(dataset.obs))

xml.Reader got no structure=… argument for StructureSpecificData


<class 'sdmx.model.v21.StructureSpecificDataSet'>
Number of Series      : 6
Number of Observations: 1399


In [31]:
# ----------------------------------------------------
# Inspect Returned Series
# ----------------------------------------------------

print("Retrieved Series:\n")

for i, series_key in enumerate(dataset.series.keys(), start=1):

    country = series_key.values["BORROWERS_CTY"].value

    print(f"{i}. {country}")

print("\nTotal Series:", len(dataset.series))

Retrieved Series:

1. US
2. BR
3. JP
4. IN
5. DE
6. CN

Total Series: 6


In [32]:
# ----------------------------------------------------
# Convert SDMX Dataset to a Panel DataFrame
# ----------------------------------------------------

rows = []

for series_key, observations in dataset.series.items():

    series_metadata = {
        dim: key_value.value
        for dim, key_value in series_key.values.items()
    }

    for obs in observations:

        row = series_metadata.copy()

        row["TIME_PERIOD"] = obs.dimension.values[0].value
        row["OBS_VALUE"] = float(obs.value)

        rows.append(row)

df_panel = pd.DataFrame(rows)

print(df_panel.shape)

df_panel.head()

(1399, 12)


,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1947-Q4,47.1
1,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q1,47.6
2,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q2,47.9
3,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q3,48.0
4,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q4,48.6


In [33]:
# ----------------------------------------------------
# Inspect Panel Dataset
# ----------------------------------------------------

print(df_panel.info())

df_panel.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1399 entries, 0 to 1398
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   BORROWERS_CTY  1399 non-null   object 
 1   TC_BORROWERS   1399 non-null   object 
 2   TC_LENDERS     1399 non-null   object 
 3   VALUATION      1399 non-null   object 
 4   UNIT_TYPE      1399 non-null   object 
 5   TC_ADJUST      1399 non-null   object 
 6   FREQ           1399 non-null   object 
 7   TITLE_TS       1399 non-null   object 
 8   UNIT_MULT      1399 non-null   object 
 9   UNIT_MEASURE   1399 non-null   object 
 10  TIME_PERIOD    1399 non-null   object 
 11  OBS_VALUE      1399 non-null   float64
dtypes: float64(1), object(11)
memory usage: 131.3+ KB
None


,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1947-Q4,47.1
1,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q1,47.6
2,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q2,47.9
3,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q3,48.0
4,US,P,A,M,770,A,Q,United States - Credit to Private non-financia...,0,367,1948-Q4,48.6


In [34]:
# ----------------------------------------------------
# Dataset Quality Checks
# ----------------------------------------------------

print("Rows              :", len(df_panel))
print("Columns           :", len(df_panel.columns))
print("Countries         :", df_panel["BORROWERS_CTY"].nunique())
print("Observations      :", df_panel["OBS_VALUE"].count())

print("\nCountries Retrieved:")

print(sorted(df_panel["BORROWERS_CTY"].unique()))

Rows              : 1399
Columns           : 12
Countries         : 6
Observations      : 1399

Countries Retrieved:
['BR', 'CN', 'DE', 'IN', 'JP', 'US']


In [35]:
# ----------------------------------------------------
# Retrieve Codelists
# ----------------------------------------------------

codelists = client.codelist().codelist

print(f"Retrieved {len(codelists)} codelists.")

Retrieved 149 codelists.


In [36]:
# ----------------------------------------------------
# Helper Function
# ----------------------------------------------------

def codelist_to_dict(codelist):
    return {
        code.id: str(code.name)
        for code in codelist.items.values()
    }